# DSAI 490 — Assignment 2 : Dates Generator
### Training Notebook (Google Colab)
+
This notebook trains all four generative models on the date generation task.
All model code lives in `.py` files in the repository — this notebook only **calls** those scripts.

**Models trained here:**
| # | Model | Type | Category |
|---|-------|------|----------|
| 1 | CVAE | Conditional Variational Autoencoder | In-course |
| 2 | CGAN | Conditional GAN (Gumbel-Softmax) | In-course |
| 3 | SeqGAN | Sequence GAN with Policy Gradient | Outside-course |
| 4 | D3PM | Discrete Absorbing Diffusion | Outside-course |

---
## 1 · Mount Google Drive
Weights are saved here so they survive Colab disconnections.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_BACKUP = '/content/drive/MyDrive/DSAI490-A2-weights'
os.makedirs(DRIVE_BACKUP, exist_ok=True)
print(f"Drive backup folder: {DRIVE_BACKUP}")

---
## 2 · Clone Repository

Replace `YOUR_TOKEN` and `YOUR_USERNAME` with your GitHub personal access token and username.
To create a token: GitHub → Settings → Developer settings → Personal access tokens → Fine-grained → repo scope.

In [ ]:
# ── FILL THESE IN ─────────────────────────────────────────────────────────
GITHUB_TOKEN    = "YOUR_TOKEN"       # your GitHub personal access token
GITHUB_USERNAME = "YOUR_USERNAME"    # your GitHub username
REPO_NAME       = "DSAI-490-A2"      # repository name
GIT_EMAIL       = "your@email.com"   # for commits
GIT_NAME        = "Your Name"        # for commits
# ──────────────────────────────────────────────────────────────────────────

REPO_URL  = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
REPO_PATH = f"/content/{REPO_NAME}"

import os
if os.path.exists(REPO_PATH):
    print("Repo already cloned — pulling latest changes...")
    os.chdir(REPO_PATH)
    !git pull
else:
    print("Cloning repository...")
    !git clone {REPO_URL} {REPO_PATH}
    os.chdir(REPO_PATH)

!git config user.email "{GIT_EMAIL}"
!git config user.name  "{GIT_NAME}"

print(f"\nWorking directory: {os.getcwd()}")
!ls

---
## 3 · Verify Environment
Colab already has PyTorch with CUDA — no conda needed.

In [ ]:
import torch, sys
print(f"Python  : {sys.version.split()[0]}")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.version.cuda}")
print(f"Device  : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

# Quick tokenizer sanity check
sys.path.insert(0, f"{REPO_PATH}/model")
from tokenizer import DateTokenizer
tok = DateTokenizer()
assert tok.vocab_size == 35
test = tok.encode("[MON] [DEC] [False] [196] 3-12-1962")
assert tok.decode(test) == "[MON] [DEC] [False] [196] 3-12-1962"
print("\n✓ Tokenizer OK  (vocab_size=35, encode/decode round-trip passed)")

---
## 4 · Helper — Backup Weights to Drive
Call `backup()` after each model finishes training.

In [ ]:
import shutil, os

def backup(model_name: str):
    src = f"{REPO_PATH}/model/{model_name}/weights"
    dst = f"{DRIVE_BACKUP}/{model_name}"
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        files = os.listdir(dst)
        print(f"✓ Backed up {model_name} weights to Drive: {files}")
    else:
        print(f"✗ No weights found for {model_name} at {src}")

print("backup() helper ready.")

---
## 5 · Train Model 1 — CVAE

**What to watch:**
- `ce` should drop steadily (decoder learning to produce correct digits)
- `kl` should stay small but non-zero (avoid posterior collapse)
- `val` should track `train` closely (no overfitting)
- `overall_csr` should climb above 50% by epoch 30

In [ ]:
os.chdir(f"{REPO_PATH}/model/vae")
!python train.py \
    --data    ../../data/data.txt \
    --epochs  60   \
    --batch   512  \
    --lr      1e-3 \
    --beta    1.0  \
    --eval_every 5

In [ ]:
backup("vae")

#### VAE — Loss curve

In [ ]:
import json, matplotlib.pyplot as plt

with open(f"{REPO_PATH}/model/vae/weights/history.json") as f:
    h = json.load(f)

epochs = range(1, len(h['train_loss']) + 1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('CVAE Training', fontsize=14)

axes[0].plot(epochs, h['train_loss'], label='train'); axes[0].plot(epochs, h['val_loss'], label='val')
axes[0].set_title('Total Loss (ELBO)'); axes[0].legend()

axes[1].plot(epochs, h['train_ce'], color='tab:orange')
axes[1].set_title('Reconstruction CE')

axes[2].plot(epochs, h['train_kl'], color='tab:green')
axes[2].set_title('KL Divergence')

for ax in axes: ax.set_xlabel('Epoch'); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{REPO_PATH}/model/vae/weights/loss_curve.png", dpi=150)
plt.show()
print("Saved loss_curve.png")

---
## 6 · Train Model 2 — CGAN (Gumbel-Softmax)

**What to watch:**
- `D(x)` should stay near 0.85–0.95
- `D(G(z))` should stay between 0.2–0.5 (not collapse to 0)
- `G` and `D` losses should fluctuate, not diverge
- `overall_csr` should start rising after epoch 30

In [ ]:
os.chdir(f"{REPO_PATH}/model/cgan")
!python train.py \
    --data        ../../data/data.txt \
    --epochs      100  \
    --batch       512  \
    --lr_g        2e-4 \
    --lr_d        2e-4 \
    --temperature 0.8  \
    --eval_every  10

In [ ]:
backup("cgan")

#### CGAN — Loss curve

In [ ]:
with open(f"{REPO_PATH}/model/cgan/weights/history.json") as f:
    h = json.load(f)

epochs = range(1, len(h['d_loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('CGAN Training', fontsize=14)

axes[0].plot(epochs, h['d_loss'], label='D loss'); axes[0].plot(epochs, h['g_loss'], label='G loss')
axes[0].set_title('Generator & Discriminator Loss'); axes[0].legend()

axes[1].plot(epochs, h['d_real'], label='D(x) — real'); axes[1].plot(epochs, h['d_fake'], label='D(G(z)) — fake')
axes[1].axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='0.5 baseline')
axes[1].set_title('Discriminator Outputs'); axes[1].legend()

for ax in axes: ax.set_xlabel('Epoch'); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{REPO_PATH}/model/cgan/weights/loss_curve.png", dpi=150)
plt.show()

---
## 7 · Train Model 3 — SeqGAN

Training runs in three automatic phases:
1. **G pretrain** (MLE) — generator learns basic date structure
2. **D pretrain** — discriminator learns to tell real from generated
3. **Adversarial** (REINFORCE) — generator improves using discriminator reward

**What to watch:**
- G pretrain CE should drop below 0.5
- D pretrain accuracy should reach 70%+
- Adversarial `D(G(z))` should stay above 0.1

In [ ]:
os.chdir(f"{REPO_PATH}/model/seqgan")
!python train.py \
    --data        ../../data/data.txt \
    --batch       512 \
    --pretrain_g  5   \
    --pretrain_d  3   \
    --adv_epochs  25  \
    --lr_g        1e-3 \
    --lr_d        1e-3 \
    --eval_every  5

In [ ]:
backup("seqgan")

#### SeqGAN — Loss curve (adversarial phase)

In [ ]:
with open(f"{REPO_PATH}/model/seqgan/weights/history.json") as f:
    h = json.load(f)

epochs = range(1, len(h['g_loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('SeqGAN Adversarial Phase', fontsize=14)

axes[0].plot(epochs, h['d_loss'], label='D loss'); axes[0].plot(epochs, h['g_loss'], label='G loss (REINFORCE)')
axes[0].set_title('G & D Loss'); axes[0].legend()

axes[1].plot(epochs, h['d_real'], label='D(x) — real'); axes[1].plot(epochs, h['d_fake'], label='D(G(z)) — fake')
axes[1].axhline(0.5, color='gray', linestyle='--', alpha=0.5)
axes[1].set_title('Discriminator Outputs'); axes[1].legend()

for ax in axes: ax.set_xlabel('Adversarial Epoch'); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{REPO_PATH}/model/seqgan/weights/loss_curve.png", dpi=150)
plt.show()

---
## 8 · Train Model 4 — D3PM (Absorbing Discrete Diffusion)

**What to watch:**
- `vlb` (VLB loss on masked positions) should decrease steadily
- `ce` (auxiliary CE on all positions) should also decrease
- A guidance sweep runs automatically at the end

**D3PM-specific evaluation at the end:**
The script prints a table of `overall_csr` vs `diversity_score` for
guidance weights w = 0, 1, 2, 4. Copy this table into your report.

In [ ]:
os.chdir(f"{REPO_PATH}/model/d3pm")
!python train.py \
    --data        ../../data/data.txt \
    --epochs      30   \
    --batch       512  \
    --lr          1e-3 \
    --T           50   \
    --lam         0.01 \
    --p_uncond    0.1  \
    --guidance    2.0  \
    --eval_every  5

In [ ]:
backup("d3pm")

#### D3PM — Loss curve + Guidance sweep

In [ ]:
with open(f"{REPO_PATH}/model/d3pm/weights/history.json") as f:
    h = json.load(f)

epochs = range(1, len(h['train_total']) + 1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('D3PM Training', fontsize=14)

axes[0].plot(epochs, h['train_total'], label='train'); axes[0].plot(epochs, h['val_total'], label='val')
axes[0].set_title('Total Loss (VLB + λ·CE)'); axes[0].legend()

axes[1].plot(epochs, h['train_vlb'], color='tab:orange')
axes[1].set_title('VLB (masked positions)')

axes[2].plot(epochs, h['train_ce'], color='tab:green')
axes[2].set_title('CE_aux (all positions)')

for ax in axes: ax.set_xlabel('Epoch'); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{REPO_PATH}/model/d3pm/weights/loss_curve.png", dpi=150)
plt.show()

# ── Guidance sweep plot ───────────────────────────────────────────────────
if h.get('guidance_sweep'):
    ws  = [float(k) for k in h['guidance_sweep']]
    csr = [h['guidance_sweep'][k]['overall_csr']     for k in h['guidance_sweep']]
    div = [h['guidance_sweep'][k]['diversity_score']  for k in h['guidance_sweep']]

    fig2, ax2 = plt.subplots(figsize=(7, 4))
    ax2.plot(ws, csr, 'o-', label='Overall CSR',     color='tab:blue')
    ax2.plot(ws, div, 's--', label='Diversity Score', color='tab:orange')
    ax2.set_xlabel('Guidance weight w'); ax2.set_ylabel('Score')
    ax2.set_title('D3PM — Guidance Scale Sweep (CSR vs Diversity)')
    ax2.legend(); ax2.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{REPO_PATH}/model/d3pm/weights/guidance_sweep.png", dpi=150)
    plt.show()
    print("Saved guidance_sweep.png")

---
## 9 · Evaluate All 4 Models

Run inference on `example_input.txt` with every model and compute the full metric suite.

In [ ]:
os.chdir(f"{REPO_PATH}/model")

models = {
    "vae":    {},
    "cgan":   {"--temperature": "0.8"},
    "seqgan": {},
    "d3pm":   {"--guidance": "2.0"},
}

for name, extra in models.items():
    extra_str = " ".join(f"{k} {v}" for k, v in extra.items())
    out_file  = f"{REPO_PATH}/model/{name}/weights/predictions.txt"
    !python predict.py \
        -i ../data/example_input.txt \
        -o {out_file} \
        --model {name} {extra_str}
    print()

In [ ]:
import sys
sys.path.insert(0, f"{REPO_PATH}/model")
from metrics import evaluate, print_metrics

summary = {}
for name in ["vae", "cgan", "seqgan", "d3pm"]:
    pred_file = f"{REPO_PATH}/model/{name}/weights/predictions.txt"
    with open(pred_file) as f:
        preds = [l.strip() for l in f if l.strip()]
    m = evaluate(preds)
    summary[name] = m
    print_metrics(m, prefix=f"{name.upper()}")

#### Side-by-side comparison

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

model_names = list(summary.keys())
metrics_to_plot = {
    "Overall CSR":    "overall_csr",
    "Day CSR":        "day_csr",
    "Month CSR":      "month_csr",
    "Leap CSR":       "leap_csr",
    "Decade CSR":     "decade_csr",
    "Validity Rate":  "validity_rate",
    "Diversity":      "diversity_score",
}

x     = np.arange(len(metrics_to_plot))
width = 0.2
colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red']

fig, ax = plt.subplots(figsize=(14, 5))
for i, (name, color) in enumerate(zip(model_names, colors)):
    vals = [summary[name].get(k, 0) for k in metrics_to_plot.values()]
    ax.bar(x + i * width, vals, width, label=name.upper(), color=color, alpha=0.85)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(metrics_to_plot.keys(), rotation=20, ha='right')
ax.set_ylabel('Score')
ax.set_ylim(0, 1.05)
ax.set_title('Model Comparison — All Evaluation Metrics')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{REPO_PATH}/model_comparison.png", dpi=150)
plt.show()
print("Saved model_comparison.png")

---
## 10 · Commit Everything to GitHub

Commits trained weights, loss curves, and the comparison chart back to your repo.

In [ ]:
os.chdir(REPO_PATH)

# Add weights and generated figures
!git add model/vae/weights/
!git add model/cgan/weights/
!git add model/seqgan/weights/
!git add model/d3pm/weights/
!git add model_comparison.png

!git status

!git commit -m "Add trained weights, loss curves, and evaluation results for all 4 models"
!git push

print("\n✓ All weights pushed to GitHub.")

---
## 11 · Quick Sample — Inspect Generated Dates

Compare what each model generates for the same set of conditions.

In [ ]:
import sys
sys.path.insert(0, f"{REPO_PATH}/model")

# Pick 5 conditions from example_input.txt
with open(f"{REPO_PATH}/data/example_input.txt") as f:
    sample_conds = [l.strip() for l in f if l.strip()][:5]

print("Conditions:")
for c in sample_conds:
    print(f"  {c}")
print()

# Load predictions for each model and show the same 5 lines
for name in ["vae", "cgan", "seqgan", "d3pm"]:
    pred_file = f"{REPO_PATH}/model/{name}/weights/predictions.txt"
    with open(pred_file) as f:
        lines = [l.strip() for l in f if l.strip()][:5]
    print(f"{'─'*60}")
    print(f"  {name.upper()}")
    for line in lines:
        print(f"    {line}")
print(f"{'─'*60}")

---
## Notes for your report

Figures saved to the repo that you can use directly:

| File | Description |
|------|-------------|
| `model/vae/weights/loss_curve.png` | ELBO, CE, KL per epoch |
| `model/cgan/weights/loss_curve.png` | G/D loss + D(x)/D(G(z)) |
| `model/seqgan/weights/loss_curve.png` | Adversarial phase G/D loss |
| `model/d3pm/weights/loss_curve.png` | VLB + CE per epoch |
| `model/d3pm/weights/guidance_sweep.png` | CSR vs Diversity across guidance weights |
| `model_comparison.png` | Side-by-side metric bar chart for all 4 models |